<a href="https://colab.research.google.com/github/Koginitiv-Analytics/supply-chain-logistics-intelligence/blob/main/Geopandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import kagglehub
import folium
from folium.plugins import HeatMap
import numpy as np
from sklearn.cluster import DBSCAN



path = kagglehub.dataset_download("shashwatwork/dataco-smart-supply-chain-for-big-data-analysis")
csv_path = os.path.join(path, "DataCoSupplyChainDataset.csv")


df = pd.read_csv(csv_path, encoding='ISO-8859-1')

print(df.head())

df.columns = df.columns.str.lower().str.replace(' ', '_')

valid_records = df.dropna(subset=['latitude', 'longitude']).copy()

geometry = [Point(xy) for xy in zip(valid_records['longitude'], valid_records['latitude'])]

gdf = gpd.GeoDataFrame(valid_records, geometry=geometry, crs="EPSG:4326")

gdf[['order_id', 'order_city', 'order_country', 'latitude', 'longitude', 'geometry']].head()



hub_lat, hub_lon = 41.8781, -87.6298
hub_point = Point(hub_lon, hub_lat)


gdf_metric = gdf.to_crs(epsg=3857)
hub_metric = gpd.GeoSeries([hub_point], crs="EPSG:4326").to_crs(epsg=3857).iloc[0]


gdf['distance_from_hub_km'] = gdf_metric.geometry.distance(hub_metric) / 1000


late_shipments = gdf[gdf['delivery_status'] == 'Late delivery']


print(f"Average Transit Distance from Hub: {gdf['distance_from_hub_km'].mean():.2f} km")
print(f"Total Late Delivery Points to Map: {len(late_shipments):,}\n")

gdf[['order_id', 'order_city', 'distance_from_hub_km', 'delivery_status']].head()


m = folium.Map(location=[hub_lat, hub_lon], zoom_start=4, tiles='CartoDB positron')


folium.Marker(
    location=[hub_lat, hub_lon],
    popup="<b>Central Distribution Hub</b>",
    icon=folium.Icon(color="black", icon="warehouse", prefix="fa")
).add_to(m)


heat_data = [[row['latitude'], row['longitude']] for _, row in late_shipments.dropna(subset=['latitude', 'longitude']).iterrows()]


HeatMap(heat_data, radius=12, blur=15, min_opacity=0.4).add_to(m)


m.save("supply_chain_risk_map.html")
m


buffer_500km = hub_metric.buffer(500 * 1000)   # 500,000 meters
buffer_1000km = hub_metric.buffer(1000 * 1000) # 1,000,000 meters


def assign_risk_zone(geom_point):
    if geom_point.within(buffer_500km):
        return 'Inner Zone (<500km)'
    elif geom_point.within(buffer_1000km):
        return 'Mid Zone (500-1000km)'
    else:
        return 'Outer Long-Haul (>1000km)'

gdf['spatial_risk_zone'] = gdf_metric.geometry.apply(assign_risk_zone)


zone_analysis = gdf.groupby('spatial_risk_zone').agg(
    total_orders=('order_id', 'count'),
    late_orders=('delivery_status', lambda x: (x == 'Late delivery').sum())
).reset_index()

zone_analysis['late_delivery_rate_%'] = (zone_analysis['late_orders'] / zone_analysis['total_orders']) * 100

print("--- Spatial Risk Zone Delay Breakdown ---")
print(zone_analysis.to_string(index=False))



buffer_500_geo = gpd.GeoSeries([buffer_500km], crs="EPSG:3857").to_crs(epsg=4326).iloc[0]
buffer_1000_geo = gpd.GeoSeries([buffer_1000km], crs="EPSG:3857").to_crs(epsg=4326).iloc[0]


coords_500 = [[y, x] for x, y in buffer_500_geo.exterior.coords]
coords_1000 = [[y, x] for x, y in buffer_1000_geo.exterior.coords]


m_buffer = folium.Map(location=[hub_lat, hub_lon], zoom_start=4, tiles='CartoDB positron')


folium.Marker(
    location=[hub_lat, hub_lon],
    popup="<b>Central Distribution Hub</b>",
    icon=folium.Icon(color="black", icon="warehouse", prefix="fa")
).add_to(m_buffer)


folium.Polygon(
    locations=coords_500,
    color="green",
    weight=2,
    fill=True,
    fill_color="green",
    fill_opacity=0.15,
    popup="<b>Inner Zone (<500km)</b><br>Delay Rate: 55.57%"
).add_to(m_buffer)


folium.Polygon(
    locations=coords_1000,
    color="orange",
    weight=2,
    fill=True,
    fill_color="orange",
    fill_opacity=0.10,
    popup="<b>Mid Zone (500-1000km)</b><br>Delay Rate: 54.71%"
).add_to(m_buffer)


m_buffer.save("geofenced_risk_zones.html")
m_buffer


late_50 = late_shipments.dropna(subset=['latitude', 'longitude']).head(50)


lines_list = []
for index, row in late_50.iterrows():
    lines_list.append(LineString([hub_point, Point(row['longitude'], row['latitude'])]))


routes_df = gpd.GeoDataFrame(late_50, geometry=lines_list, crs="EPSG:4326")


routes_df['dist_km'] = routes_df.to_crs(epsg=3857).geometry.length / 1000


my_map = folium.Map(location=[hub_lat, hub_lon], zoom_start=4, tiles='CartoDB dark_matter')


folium.Marker(
    location=[hub_lat, hub_lon],
    popup="Chicago Hub",
    icon=folium.Icon(color="red", icon="info-sign")
).add_to(my_map)


for index, row in routes_df.iterrows():
    start_end = [[hub_lat, hub_lon], [row['latitude'], row['longitude']]]

    folium.PolyLine(
        locations=start_end,
        color="red",
        weight=2,
        opacity=0.5
    ).add_to(my_map)

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color="blue",
        popup=row['order_city']
    ).add_to(my_map)


my_map.save("routes.html")
my_map

Using Colab cache for faster access to the 'dataco-smart-supply-chain-for-big-data-analysis' dataset.
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                         3                              4   
1  TRANSFER                         5                              4   
2      CASH                         4                              4   
3     DEBIT                         3                              4   
4   PAYMENT                         2                              4   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk  Category Id   Category Name Customer City  ...  \
0                  